# Desarrollo de distintos modelos.

## RANDOM FOREST

In [7]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("dataset_final.csv")
groups = df['OS'].map(str) + "_" + df['Ejecution'].map(str)
df['OS'] = df['OS'].map({'Windows': 1, 'Linux': 0})

imputer = SimpleImputer(strategy='mean')

gss = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(df, df['Infected'], groups))
train = df.iloc[train_idx]
test  = df.iloc[test_idx]

X_train = train.drop(columns=['Infected', 'Ejecution'])
y_train = train['Infected']

X_test = test.drop(columns=['Infected', 'Ejecution'])
y_test = test['Infected']

X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

rf = RandomForestClassifier(
    n_estimators=200,        
    max_depth=None,         
    random_state=42,
    n_jobs=-1                
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]
aux = pd.DataFrame(X_train, columns=df.drop(columns=['Infected', 'Ejecution']).columns)
importances = pd.Series(rf.feature_importances_, index=aux.columns).sort_values(ascending=False)
print(importances.head(10))


print("=== Matriz de confusión ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob))


Avg_num_writes                                      0.139179
Avg_num_reads                                       0.098297
procceses_with_more_writes_than_reads_percentage    0.096382
Create_filesystem_events                            0.053413
Avg_bytes_read                                      0.045139
Directory_event_ratio                               0.040853
EventID_entropy                                     0.038824
Avg_bytes_written                                   0.035362
Unique_path_count                                   0.031047
different_parents_per_processes                     0.028693
dtype: float64
=== Matriz de confusión ===
[[17  0]
 [ 1 39]]

=== Reporte de clasificación ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        17
           1       1.00      0.97      0.99        40

    accuracy                           0.98        57
   macro avg       0.97      0.99      0.98        57
weighted avg       0.9

## XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

df = pd.read_csv("dataset_final.csv")
groups = df['OS'].map(str) + "_" + df['Ejecution'].map(str)
df['OS'] = df['OS'].map({'Windows': 1, 'Linux': 0})

imputer = SimpleImputer(strategy='mean')

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

"""""
gkf = GroupKFold(n_splits=5)
df1 = df.drop(columns=['Infected', 'Ejecution'])
scores = cross_val_score(
    xgb_model,
    df1, df['Infected'],
    groups=groups,   # tu ID de ejecución
    cv=gkf,
    scoring="f1"
)

print("Scores XGBoost:", scores)
print("Mean:", scores.mean())
"""

gss = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(df, df['Infected'], groups))
train = df.iloc[train_idx]
test  = df.iloc[test_idx]

X_train = train.drop(columns=['Infected', 'Ejecution'])
y_train = train['Infected']

X_test = test.drop(columns=['Infected', 'Ejecution'])
y_test = test['Infected']

X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]
aux = pd.DataFrame(X_train, columns=df.drop(columns=['Infected', 'Ejecution']).columns)
importances = pd.Series(xgb_model.feature_importances_, index=aux.columns).sort_values(ascending=False)
print(importances.head(10))


print("=== Matriz de confusión ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

Avg_num_reads                                       0.137619
Delete_filesystem_events                            0.128807
Avg_num_writes                                      0.104051
Unique_path_count                                   0.074281
Num_processes                                       0.053222
CPU_user                                            0.048380
procceses_with_more_writes_than_reads_percentage    0.046933
CPU_system                                          0.045324
different_parents_per_processes                     0.042152
Avg_bytes_read                                      0.038780
dtype: float32
=== Matriz de confusión ===
[[17  0]
 [ 1 39]]

=== Reporte de clasificación ===
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        17
           1       1.00      0.97      0.99        40

    accuracy                           0.98        57
   macro avg       0.97      0.99      0.98        57
weighted avg       0.9

## LIGHTGBM

In [20]:
from lightgbm import LGBMClassifier

df = pd.read_csv("dataset_final.csv")
groups = df['OS'].map(str) + "_" + df['Ejecution'].map(str)
df['OS'] = df['OS'].map({'Windows': 1, 'Linux': 0})

imputer = SimpleImputer(strategy='mean')

lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    num_leaves=7,
    min_data_in_leaf=2,
    feature_fraction=1.0,
    bagging_fraction=1.0,
    bagging_freq=0,
    min_gain_to_split=0.0,
    vervose=-1
)
"""""
gkf = GroupKFold(n_splits=5)
df1 = df.drop(columns=['Infected', 'Ejecution'])
scores = cross_val_score(
    lgbm_model,
    df1, df['Infected'],
    groups=groups,
    cv=gkf,
    scoring="f1"
)

print("Scores LightGBM:", scores)
print("Mean:", scores.mean())
"""
gss = GroupShuffleSplit(test_size=0.3, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(df, df['Infected'], groups))
train = df.iloc[train_idx]
test  = df.iloc[test_idx]

X_train = train.drop(columns=['Infected', 'Ejecution'])
y_train = train['Infected']

X_test = test.drop(columns=['Infected', 'Ejecution'])
y_test = test['Infected']

X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

lgbm_model.fit(X_train, y_train)

y_pred = lgbm_model.predict(X_test)
y_prob = lgbm_model.predict_proba(X_test)[:, 1]
aux = pd.DataFrame(X_train, columns=df.drop(columns=['Infected', 'Ejecution']).columns)
importances = pd.Series(lgbm_model.feature_importances_, index=aux.columns).sort_values(ascending=False)
print(importances.head(10))


print("=== Matriz de confusión ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== Reporte de clasificación ===")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

[LightGBM] [Warning] Unknown parameter: vervose
[LightGBM] [Warning] min_data_in_leaf is set=2, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=2
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
[LightGBM] [Warning] bagging_freq is set=0, subsample_freq=0 will be ignored. Current value: bagging_freq=0
[LightGBM] [Warning] Unknown parameter: vervose
[LightGBM] [Warning] min_data_in_leaf is set=2, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=2
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0

c:\Users\arman\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\arman\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
